# Grad-CAM 3D + độ nhạy theo thì

> **Research Use Only (RUO)** — chưa kiểm định lâm sàng, không dùng chẩn đoán.

**Không train gì.** Nạp `best_fold_N.pt`, chạy forward + backward cho 4 ca demo, lưu
bản đồ chú ý và vector độ nhạy theo thì.

**Mỗi ca dùng model của chính fold chứa nó ở tập val** — model chưa từng thấy ca đó
khi train, nhất quán với nguyên tắc out-of-fold của cả dự án.

**Bản đồ sống trong không gian crop 112×112×32**, không phải lát gốc 480×480. Mô hình
chưa từng thấy ảnh gốc: nó nhận khối đã cắt bám tổn thương và căn từng thì. Phủ bản
đồ lên ảnh gốc sẽ là một tuyên bố sai về những gì mô hình nhìn thấy.

⚠️ **Cổng B là cổng quan trọng nhất.** DenseNet121 hạ mẫu 5 lần; tầng cuối rất có thể
chỉ còn Z = 1, nghĩa là bản đồ giống hệt nhau ở cả 32 lát. Nó vẫn phóng lên mượt và
vẫn trông thuyết phục, nên lỗi đó **không tự lộ ra**. Cổng B đo hình dạng thật rồi mới
cho chạy.

⚠️ **Mặc định `MODE = "hires"` (HiResCAM), không phải Grad-CAM gốc.** Dense block của
MONAI là concat các đầu ra conv nên đặc trưng **có giá trị âm**, vi phạm giả định của
Grad-CAM gốc — và ở ca `MR207769` nó cho bản đồ toàn 0 (S-095). HiResCAM nhân theo
từng phần tử nên đúng cho đặc trưng có dấu.

**Ngân sách:** vài phút GPU. Cần mount hai dataset: cache E4 và `best-weights`.

## 0. Bootstrap

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/hdtruong802/liver-mri-3d-classifier.git"

# ---- THAM SỐ ---------------------------------------------------------------
FOLDS = [1, 2, 3, 4, 5]
LAYER = "denseblock3"        # Cổng B sẽ xác nhận tầng này còn đủ độ phân giải
MODE = "hires"               # "hires" = HiResCAM. "gradcam" = bản gốc, CHO BẢN ĐỒ
                             # TOÀN 0 ở một số ca trên DenseNet (S-095).
DEMO_CASES = ["MR170828", "MR207769", "MR113627", "MR127280"]
# ----------------------------------------------------------------------------

REPO = Path("/kaggle/working/repo")
os.chdir("/kaggle/working")
subprocess.run(["rm", "-rf", str(REPO)], check=False)
subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)
sys.path.insert(0, str(REPO))
os.chdir(REPO)

for name in [m for m in list(sys.modules) if m == "src" or m.startswith("src.")]:
    del sys.modules[name]

print("repo commit:", subprocess.run(
    ["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True,
).stdout.strip())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "monai"], check=True)

os.environ["LLDMMRI_OUTPUT_DIR"] = "/kaggle/working/runs/gradcam"
os.environ.pop("LLDMMRI_DATA_ROOT", None)

from src.utils.io import load_yaml, repo_root  # noqa: E402

assert repo_root() == REPO.resolve(), "src/ nạp từ chỗ khác — restart kernel"

CFG_PATH = REPO / "configs" / "baseline_3dpatch.yaml"
CFG = load_yaml(CFG_PATH)
print("\nca sẽ chạy:", DEMO_CASES)

## 1. Cache và checkpoint

Nhận diện bằng **nội dung**, không bằng tên dataset (WORKLOG S-082…S-086).

In [ ]:
INPUT_ROOT = Path("/kaggle/input")

# Cache E4 nhận diện bằng NỘI DUNG `cache_meta.json`, không bằng tên dataset. Tên do
# người upload đặt và đã lệch một lần rồi (`lld-mmri-lesion-tight/cache_lesion_tight`
# chứ không phải `lld-mmri-e4-per-phase` như đoán ở S-080). Ba khoá này là thứ phân
# biệt E4 với mọi cache trước đó.
E4_KEYS = {
    "align_phases": "per_phase",          # <- phân biệt E4 với E3
    "target_size": [112, 112, 32],        # <- phân biệt E3/E4 với E0/E1
    "crop_mode": "lesion_tight",          # <- phân biệt E1+ với E0
}

# Tên file checkpoint. KHÔNG kèm thư mục cha — độ sâu do `rglob` lo, xem bên dưới.
#   A) best_fold_1.pt ... best_fold_5.pt   <- dataset "best weights"
#   B) fold_1/best.pt ...                   <- gói thẳng từ output run
CKPT_NAMES = ["best_fold_{f}.pt", "best.pt"]

# ---------------------------------------------------------------------------
# KHÔNG hardcode độ sâu. Kaggle mount ở `/kaggle/input/datasets/<user>/<slug>/...`
# chứ không phải `/kaggle/input/<slug>/...` như mọi notebook trước giả định, và độ
# sâu đó có thể đổi tiếp. Dò theo TÊN FILE mốc, sâu bao nhiêu cũng thấy. Đây là lần
# thứ tư sửa cùng một lớp lỗi (S-081 → S-084); nguyên nhân gốc luôn là một giả định
# về hình dạng đường dẫn.
#
# MỘT lượt `os.walk` duy nhất thu hết mọi thứ cần. Không dùng nhiều `rglob` riêng:
# dataset gốc là 83.7GB / ~4000 file trên ổ mạng, và mỗi `rglob` là một lượt duyệt
# toàn cây — 11 lượt thì chờ rất lâu mà chẳng được gì thêm.
# ---------------------------------------------------------------------------
import json as _json
import os as _os
import re as _re

_cfg_data = load_yaml(REPO / "configs" / "data.yaml")
_ann_name = Path(_cfg_data["annotation_rel"]).name

interesting = {}       # thư mục -> số .npz/.pt/meta, để in bảng chẩn đoán
meta_paths = []        # cache_meta.json tìm được
ckpt_paths = []        # mọi file .pt tên best*.pt
_ann = []              # file annotation của dữ liệu gốc

for dirpath, dirnames, filenames in _os.walk(INPUT_ROOT):
    dirnames[:] = [x for x in dirnames if x not in (".cache", ".git")]  # rác tải HF
    here = {"npz": 0, "pt": 0, "meta": 0}
    for name in filenames:
        full = Path(dirpath) / name
        if name.endswith(".npz"):
            here["npz"] += 1
        elif name.endswith(".pt"):
            here["pt"] += 1
            if name.startswith("best"):
                ckpt_paths.append(full)
        elif name == "cache_meta.json":
            here["meta"] += 1
            meta_paths.append(full)
        elif name == _ann_name:
            _ann.append(full)
    if any(here.values()):
        interesting[Path(dirpath)] = here


def read_caches(paths):
    out = []
    for p in sorted(paths):
        try:
            out.append((p.parent, _json.loads(p.read_text("utf-8"))))
        except Exception as exc:  # noqa: BLE001 - chỉ để báo cáo, không nuốt lỗi thật
            out.append((p.parent, {"__loi__": repr(exc)}))
    return out


def matches_e4(meta):
    return all(meta.get(k) == v for k, v in E4_KEYS.items())


def pick_checkpoints(paths, folds):
    """{fold: đường dẫn}. Ưu tiên `best_fold_N.pt`; `best.pt` thì suy fold từ thư
    mục cha (`fold_3/best.pt`). Không suy được thì bỏ, không đoán bừa."""
    out = {}
    for fold in folds:
        hits = [p for p in sorted(paths) if p.name == f"best_fold_{fold}.pt"]
        if not hits:
            hits = [
                p for p in sorted(paths)
                if p.name == "best.pt"
                and (m := _re.search(r"fold_?(\d+)", p.parent.name))
                and int(m.group(1)) == fold
            ]
        if hits:
            out[fold] = hits[0]
    return out


print(f"=== Thư mục có dữ liệu dưới {INPUT_ROOT} ===")
for d in sorted(interesting)[:25]:
    c = interesting[d]
    print(f"  {d}\n      {' · '.join(f'{c[k]} {k}' for k in ('npz', 'pt', 'meta') if c[k])}")
if not interesting:
    print("  (trống — chưa mount dataset nào có .npz/.pt)")

print(f"\n=== Dữ liệu gốc ({_ann_name}) ===")
for p in sorted(_ann)[:5]:
    print(f"  ✓ {p.parent.parent}")
if not _ann:
    print("  KHÔNG thấy — chỉ cần nếu phải build cache (xem ngay dưới)")

caches = read_caches(meta_paths)
print(f"\n=== {len(caches)} cache có cache_meta.json ===")
for path, meta in caches:
    mark = "✓ E4" if matches_e4(meta) else "  --"
    print(
        f"  {mark}  {path}\n"
        f"        crop={meta.get('crop_mode')} size={meta.get('target_size')} "
        f"align={meta.get('align_phases')}"
    )

e4 = [p for p, m in caches if matches_e4(m)]
if e4:
    CACHE_DIR = e4[0]
    BUILD_NEEDED = False
else:
    # Thư mục có nhiều .npz nhưng KHÔNG có meta: không dùng được, và phải nói rõ vì sao.
    # Hình dạng mảng cho biết target_size, nhưng KHÔNG cho biết `align_phases` —
    # E3 (reference) và E4 (per_phase) có cùng shape [8,112,112,32]. Nhận nhầm E3
    # thành E4 sẽ cho ra một bảng kết quả sai mà trông hoàn toàn hợp lý.
    for d, c in sorted(interesting.items()):
        if c["npz"] > 100 and not c["meta"]:
            print(f"\n⚠ {d} có {c['npz']} file .npz nhưng KHÔNG có cache_meta.json.")
            print("  Không dùng được: shape cho biết target_size nhưng KHÔNG phân biệt được")
            print("  E3 (align=reference) với E4 (align=per_phase) — hai cái cùng shape.")
    BUILD_NEEDED = True
    CACHE_DIR = Path("/kaggle/working/cache_e4")
    if _ann:
        print("\n=> sẽ BUILD lại cache E4 (~26 phút). Dữ liệu gốc đã có ✓")
    else:
        print(
            "\n=> CẦN BUILD cache E4 nhưng CHƯA MOUNT dữ liệu gốc.\n"
            f"   Mount dataset chứa {_cfg_data['annotation_rel']} "
            f"(ứng viên: {_cfg_data.get('data_root_candidates')}),\n"
            "   rồi chạy lại từ cell này."
        )

CKPTS = pick_checkpoints(ckpt_paths, FOLDS)
thieu = [f for f in FOLDS if f not in CKPTS]
assert not thieu, (
    f"không thấy checkpoint cho fold {thieu}.\n"
    f"Đã dò theo tên {CKPT_NAMES} ở MỌI độ sâu dưới {INPUT_ROOT}.\n"
    f"Tìm được: { {f: str(p) for f, p in CKPTS.items()} }"
)
print("\ncache:      ", CACHE_DIR, "(CHƯA CÓ — sẽ build ở cell dưới)" if BUILD_NEEDED else "")

# 5 file cùng kiến trúc nên cùng kích thước — kích thước KHÔNG chứng minh chúng khác
# nhau. Băm để chắc không phải một file bị chép 5 lần với 5 cái tên.
import hashlib

print("checkpoint:")
digests = {}
for f in FOLDS:
    p = CKPTS[f]
    h = hashlib.sha256(p.read_bytes()).hexdigest()[:16]
    digests[f] = h
    print(f"  fold {f}: {p.name}  {p.stat().st_size / 2**20:.1f} MB  sha256 {h}")
assert len(set(digests.values())) == len(FOLDS), f"có checkpoint trùng nhau: {digests}"

# Đối chiếu với mã băm đo ở máy local (WORKLOG S-081). Khác => file khác bản.
LOCAL_SHA = {
    1: "2e1f3e1ad477ad59", 2: "30a8eb9ee221d453", 3: "00c133e031bdf8fe",
    4: "3fe18f1eb3de4431", 5: "d61cc7ed94b8ebf0",
}
lech = {f: (digests[f], LOCAL_SHA[f]) for f in FOLDS if f in LOCAL_SHA and digests[f] != LOCAL_SHA[f]}
if lech:
    print(f"\n⚠ mã băm khác bản local: {lech}")
    print("  Không tự dừng — nhưng nếu bạn không cố ý đổi checkpoint thì hãy dừng lại xem.")

## 1b. Build cache nếu chưa có (~26 phút)

In [ ]:
if BUILD_NEEDED:
    from src.utils.io import resolve_data_root

    # Data root khai ở configs/data.yaml, KHÔNG ở preprocess_*.yaml — file preprocess
    # chỉ có tham số tiền xử lý. `build_cache` cũng đọc data.yaml (xem hàm main của
    # nó). Kiểm trước ở đây chỉ để fail nhanh, thay vì chết giữa job 26 phút.
    cfg_data = load_yaml(REPO / "configs" / "data.yaml")
    try:
        data_root = resolve_data_root(cfg_data)
    except Exception as exc:
        # RuntimeError chứ không SystemExit: SystemExit làm IPython lỗi khi dựng
        # traceback và che mất thông báo thật bằng một trang lỗi của chính nó.
        data_root, exc_msg = None, str(exc)
    else:
        exc_msg = None

    # `resolve_data_root` trả về `config['data_root']` mà KHÔNG xác minh khi mọi cách
    # dò đều trượt (src/utils/io.py:219-225). Trên Kaggle nó sẽ là `data/lldmmridataset`
    # tương đối, không tồn tại — và job 26 phút sẽ chết giữa chừng. Xác minh ở đây.
    ann = (data_root / cfg_data["annotation_rel"]) if data_root else None
    if ann is None or not ann.exists():
        raise RuntimeError(
            f"Không tìm thấy dữ liệu LLD-MMRI gốc.\n"
            f"  resolve_data_root -> {data_root}"
            + (f" (lỗi: {exc_msg})" if exc_msg else f", nhưng {ann} không tồn tại")
            + f"\n  Cần mount dataset chứa {cfg_data['annotation_rel']}.\n"
            f"  Ứng viên khai trong configs/data.yaml: {cfg_data.get('data_root_candidates')}"
        ) from None
    print("data root:", data_root, "✓")

    os.environ["LLDMMRI_CACHE_DIR"] = str(CACHE_DIR)
    rc = subprocess.run(
        [sys.executable, "-m", "src.preprocess.build_cache",
         "--config", "configs/preprocess_e4.yaml"],
        cwd=REPO,
    ).returncode
    assert rc == 0, "build cache thất bại"
    print("build xong:", CACHE_DIR)
else:
    print("bỏ qua build — đã có cache E4:", CACHE_DIR)

os.environ["LLDMMRI_CACHE_DIR"] = str(CACHE_DIR)

## Cổng A ⚠️ — cache có đúng là E4 không

In [ ]:
import json

meta = json.loads((Path(os.environ["LLDMMRI_CACHE_DIR"]) / "cache_meta.json").read_text("utf-8"))

# Dùng lại E4_KEYS của cell trên, không chép ra bản thứ hai — hai bản sẽ trôi khỏi nhau.
for key, want in E4_KEYS.items():
    got = meta.get(key)
    assert got == want, f"cache SAI: {key} = {got!r}, cần {want!r}. Đây không phải cache E4."
assert meta["lesion_tight"]["source"] == "mask", "phải cắt theo mask, không phải bbox"

n_npz = len(list(Path(os.environ["LLDMMRI_CACHE_DIR"]).glob("*.npz")))
assert n_npz >= 498, f"chỉ có {n_npz} ca, cần 498 — cache chưa build xong"
print(f"cache_meta khớp E4 ✓ · {n_npz} ca · commit {meta.get('git_commit')}")

## Cổng B ⚠️⚠️ — tầng nào còn đủ độ phân giải

Bảng dưới in hình dạng đầu ra thật của từng tầng. **Nếu tầng đã chọn có chiều nào bằng
1 thì dừng lại và chọn tầng nông hơn** — đừng chạy tiếp. Một bản đồ hằng số theo Z
phóng lên 32 lát trông không khác gì một bản đồ thật.

Đánh đổi khi lùi về tầng nông: sắc nét hơn nhưng ít mang tính lớp hơn. Ghi lại tầng đã
dùng cùng kết quả để người đọc tự đánh giá.

In [ ]:
import torch

from src.models import build_model
from src.xai.gradcam import CANDIDATE_LAYERS, feature_layer_shapes

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

probe = build_model(CFG["model"])
target_size = meta["target_size"]          # từ cache_meta, không hardcode
shapes = feature_layer_shapes(probe, (1, CFG["model"]["in_channels"], *target_size))

print(f"\nĐầu vào: {tuple(target_size)}\n")
print(f"{'tầng':<16}{'kênh':>7}{'không gian':>18}   dùng được?")
print("-" * 58)
for name in CANDIDATE_LAYERS:
    if name not in shapes:
        continue
    spatial = shapes[name][2:]
    ok = "OK" if all(v >= 2 for v in spatial) else "KHÔNG — có chiều = 1"
    mark = " <-- đang chọn" if name == LAYER else ""
    print(f"{name:<16}{shapes[name][1]:>7}{str(spatial):>18}   {ok}{mark}")

chosen = shapes[LAYER][2:]
assert all(v >= 2 for v in chosen), (
    f"tầng {LAYER} cho bản đồ {chosen} — có chiều bằng 1 nên bản đồ là hằng số theo "
    "chiều đó. Chọn tầng nông hơn ở bảng trên rồi chạy lại cell 0."
)
print(f"\n✓ {LAYER}: bản đồ {chosen}, phóng lên {tuple(target_size)}")
print(f"  hệ số phóng: {[round(t / c, 1) for t, c in zip(target_size, chosen)]}")
del probe

## 2. Chạy cho từng ca

Nạp cache của ca, tìm fold có ca đó ở tập **val**, nạp đúng checkpoint của fold ấy.

In [ ]:
import numpy as np

from src.utils.ids import normalize_pid
from src.xai.gradcam import grad_cam_3d, phase_importance

CACHE = Path(os.environ["LLDMMRI_CACHE_DIR"])
PHASES = [p["file"] for p in load_yaml(REPO / "configs" / "data.yaml")["phases"]]
REF_INDEX = PHASES.index("C+V")

# Đọc thẳng file split thay vì qua `Splits`: lớp đó chỉ có `trainval_keys`/`test_keys`,
# không có "fold nào chứa ca này ở val" — mà đó mới là thứ cần, vì phải dùng đúng model
# CHƯA train trên ca đang giải thích.
VAL_KEYS = {
    fold: {
        normalize_pid(line.split()[0])
        for line in (REPO / "splits" / f"val_fold{fold}.txt").read_text().splitlines()
        if line.strip()
    }
    for fold in FOLDS
}


def fold_of(pid: str) -> int:
    key = normalize_pid(pid)
    for fold, keys in VAL_KEYS.items():
        if key in keys:
            return fold
    raise KeyError(f"{pid} không nằm ở val của fold nào — có phải ca test-104 không?")


def load_case(pid: str) -> tuple[np.ndarray, int]:
    """Khối 8 kênh + nhãn thật, lấy từ chính cache mà model được train trên đó."""
    hits = sorted(CACHE.glob(f"{pid}*.npz")) or sorted(CACHE.glob(f"*{normalize_pid(pid)}*.npz"))
    assert hits, f"không thấy cache của {pid} trong {CACHE}"
    data = np.load(hits[0])
    return np.asarray(data["image"], dtype=np.float32), int(data["label"])


def to_uint8(channel: np.ndarray) -> np.ndarray:
    lo, hi = float(channel.min()), float(channel.max())
    return ((channel - lo) / max(hi - lo, 1e-6) * 255).astype(np.uint8)


OUT = Path("/kaggle/working/gradcam")
OUT.mkdir(parents=True, exist_ok=True)

for pid in DEMO_CASES:
    fold = fold_of(pid)
    model = build_model(CFG["model"]).to(DEVICE)
    model.load_state_dict(torch.load(CKPTS[fold], map_location=DEVICE)["model"])

    # BẮT BUỘC: `build_model` trả về model ở chế độ TRAIN (mặc định của nn.Module).
    # Bỏ dòng này thì (a) dropout vẫn bật -> xác suất đổi mỗi lần chạy, (b) BatchNorm
    # dùng thống kê của batch, mà batch ở đây là 1 mẫu -> mỗi kênh bị chuẩn hoá bằng
    # chính nó, khác hẳn running stats đã học. Cả hai làm `pred` sai, và Grad-CAM khi
    # đó giải thích một lớp mà model không thật sự đoán (WORKLOG S-096).
    model.eval()
    assert not any(m.training for m in model.modules()), "còn module ở chế độ train"

    array, true = load_case(pid)
    volume = torch.from_numpy(array)[None].to(DEVICE)
    with torch.no_grad():
        probs = torch.softmax(model(volume), dim=1)[0].cpu().numpy()
    pred = int(probs.argmax())

    # `grad_cam_3d` tự nổ kèm số chẩn đoán nếu bản đồ suy biến — không cần assert ở đây.
    cam_pred, native = grad_cam_3d(model, volume, pred, LAYER, mode=MODE)

    payload = {
        "cam_pred": cam_pred.astype(np.float16),
        "crop_ref": to_uint8(array[REF_INDEX]),
        "phase_importance": phase_importance(model, volume, pred),
        "pred_index": np.int64(pred),
        "true_index": np.int64(true),
        "fold": np.str_(f"fold_{fold}"),
        "layer": np.str_(f"{LAYER} · {MODE}"),
        "cam_native_shape": np.asarray(native, dtype=np.int64),
    }
    if pred != true:
        # Đoán sai: bản đồ của lớp THẬT cho thấy mô hình lẽ ra phải nhìn vào đâu.
        cam_true, _ = grad_cam_3d(model, volume, true, LAYER, mode=MODE)
        payload["cam_true"] = cam_true.astype(np.float16)

    np.savez_compressed(OUT / f"{pid}.npz", **payload)
    top = int(payload["phase_importance"].argmax())
    print(
        f"{pid}  fold {fold}  thật={true} đoán={pred} p={probs[pred]:.3f}"
        f"{'  SAI' if pred != true else '     '}"
        f"  cam max={cam_pred.max():.3f}  thì nổi nhất: {PHASES[top]}"
    )
    del model
    torch.cuda.empty_cache()

## 3. Gói mang về

Ở máy local đặt vào `runs/E4_per_phase_results/gradcam/`, backend tự nhận.

In [ ]:
import shutil

total = sum(f.stat().st_size for f in OUT.rglob("*") if f.is_file())
print(f"{OUT}: {total / 2**20:.1f} MiB")
for f in sorted(OUT.glob("*.npz")):
    print(f"  {f.name}  {f.stat().st_size / 2**10:.0f} KiB")

shutil.make_archive("/kaggle/working/gradcam", "zip", OUT)
print("""
→ /kaggle/working/gradcam.zip
⚠ Giải nén CHỈ MỘT LỚP vào runs/E4_per_phase_results/gradcam/ (.npz bản thân là zip).
""")